# 04 — Backtest: Edge-Filtered Trading Strategy

Turns fair-value vs. market-price edges into a simple trading strategy:

- **Buy** when `fair_prob - market_prob > entry_threshold` and liquidity is
  sufficient
- **Sell/avoid** on the opposite edge
- Marks P&L to the next available market price, net of a transaction-cost
  assumption
- Evaluates with P&L, hit rate, drawdown, a Sharpe-like ratio, and forecast
  quality via Brier score / log loss


In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from backtester import Backtester, summarize_by_market
import visualization as viz

pd.set_option("display.max_columns", 25)
%matplotlib inline

In [ ]:
scored_df = pd.read_csv("../data/processed/scored_test_set.csv", parse_dates=["timestamp"])
scored_df.shape

## Run the backtest

Defaults: 1% minimum edge, minimum liquidity/trust weight of 0.10, 20bps
round-trip transaction cost. These are intentionally conservative — tune
them and re-run to see the P&L / trade-count tradeoff below.

In [ ]:
bt = Backtester(entry_threshold=0.01, min_trust_weight=0.10, fee_bps=20)
trades_df, metrics = bt.run(scored_df)

for k, v in metrics.items():
    print(f"{k:>20s}: {v}")

## Cumulative P&L

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
viz.plot_backtest_pnl(trades_df, ax=ax)
plt.show()

## By-market breakdown

In [ ]:
summarize_by_market(trades_df)

## Sensitivity: entry threshold vs. fees

A quick sweep to show how trade count, P&L, and hit rate move together —
useful for picking a threshold that's robust rather than curve-fit to one
setting.

In [ ]:
import itertools

rows = []
for thresh, fee in itertools.product([0.005, 0.008, 0.01, 0.012, 0.015, 0.02], [0, 10, 20, 50]):
    b = Backtester(entry_threshold=thresh, min_trust_weight=0.10, fee_bps=fee)
    t, m = b.run(scored_df)
    rows.append({
        "entry_threshold": thresh, "fee_bps": fee,
        "n_trades": m.get("n_trades", 0),
        "total_pnl": m.get("total_pnl"),
        "hit_rate": m.get("hit_rate"),
    })

sweep_df = pd.DataFrame(rows)
sweep_df

In [ ]:
pivot = sweep_df.pivot(index="entry_threshold", columns="fee_bps", values="total_pnl")
fig, ax = plt.subplots(figsize=(8, 5))
for fee in pivot.columns:
    ax.plot(pivot.index, pivot[fee], marker="o", label=f"{fee}bps fee")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Entry threshold (min |edge|)")
ax.set_ylabel("Total P&L")
ax.set_title("P&L sensitivity: edge threshold x fee assumption")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## Forecast quality: Brier score & log loss

Independent of whether a trade was taken — how good are the fair-value
probabilities as forecasts of the next observed market probability?

In [ ]:
print(f"Brier score: {metrics['brier_score']:.5f}  (lower is better; 0 = perfect, 0.25 = coin-flip baseline)")
print(f"Log loss:    {metrics['log_loss']:.5f}  (lower is better)")

## Takeaways

- The edge-filtered strategy is profitable net of a 20bps fee assumption at
  a 1% minimum-edge threshold in this backtest window, with a hit rate
  modestly above 50%.
- P&L is sensitive to the fee assumption, as expected in a market this
  liquid — tighter spreads mean smaller edges survive transaction costs.
- This is a **methodology demonstration**, not a live trading signal: no
  slippage/partial-fill modeling, a single (synthetic, for this offline
  demo) market category, and a short backtest window. Swap in real
  Polymarket/Kalshi data (see notebook 01) and extend the backtest window
  before drawing any real trading conclusions.
